# Week 1 — SELECT, WHERE, ORDER BY, LIMIT: Filtered Exploration
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Apply SELECT, WHERE, ORDER BY, and LIMIT independently to answer business questions (group work)

Today is a walkthrough day. We'll apply Wednesday's clauses to four tables you
haven't touched yet — `order_items`, `order_payments`, `customers`, and
`order_reviews` — so the patterns feel familiar before you break into your
**group exercise**: five real business questions, 10 minutes each, answers
verified together as a class at the end.

### Setup — run this cell first

Same setup as Wednesday: it loads the 8 Olist tables into a file-based SQLite
database and connects the `%%sql` cell magic to it. Run it once at the start
of the session and leave it alone — every query below (and every query in the
group exercise) depends on it having run successfully.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

Wednesday you learned to filter and sort the `orders` table (99,441 rows).
Olist's database has several other tables built the same way: `order_items`
holds 112,650 rows (one per line item in an order), `order_payments` holds
103,886 rows (one per payment made against an order), `customers` holds
99,441 rows (one per customer), and `order_reviews` holds 99,224 rows (one
per submitted review). None of that changes how `SELECT`, `WHERE`, `ORDER BY`,
and `LIMIT` work — the same four clauses answer business questions on any
table with rows and columns. In a few minutes you'll put that to the test in
today's **group exercise**, where each of your five questions targets one of
these tables. First, let's walk through one query on each table together.

## 1. Revisiting `ORDER BY` and `LIMIT` — now on `order_items`

`order_items` is new today: 112,650 rows, one row per line item within an
order (an order with three products has three rows here, not one). Everything
you learned about `ORDER BY` and `LIMIT` on Wednesday transfers directly —
sort ascending with `ASC` (the default if you omit it) or descending with
`DESC`, then cap how many rows come back with `LIMIT`. Below we sort by
`price` ascending to see Olist's cheapest individual items; in the group
exercise you'll flip the direction to find the most expensive ones.

In [ ]:
%%sql
-- Cheapest individual items first — same ORDER BY / LIMIT pattern, new table
SELECT order_id, price, freight_value
FROM order_items
ORDER BY price ASC
LIMIT 10

## 2. `WHERE` on `order_payments` — filtering by payment method

`order_payments` records every payment made against an order — 103,886 rows
in total, more than the number of orders, because some orders are split
across multiple payments or installments. Brazilian customers pay with
`credit_card`, `boleto` (a bank-slip method), `voucher`, or `debit_card`,
stored in the `payment_type` column. It's the exact same
`WHERE column = 'value'` pattern you used on `order_status` on Wednesday —
just point it at a different column and value.

In [ ]:
%%sql
-- How many payments were made by credit card?
SELECT COUNT(*) AS total
FROM order_payments
WHERE payment_type = 'credit_card'   -- Expected: 76,795

## 3. `WHERE` on `customers` — filtering by geography

`customers` holds 99,441 rows, one per customer, with a `customer_state`
column storing Brazil's 2-letter state codes — `SP`, `RJ`, `MG`, and so on.
"How many of our customers are in this region?" is one of the most common
real business questions you'll be asked, and it's answered with the exact
same `WHERE` filter you already know. Below we peek at customers based in
`RJ` (Rio de Janeiro) just to see the shape of the rows — in the group
exercise you'll count customers in a different state.

In [ ]:
%%sql
-- A quick look at customers based in Rio de Janeiro (RJ) — display only
SELECT customer_id, customer_city, customer_state
FROM customers
WHERE customer_state = 'RJ'
LIMIT 10

## 4. `WHERE` on `order_reviews` — filtering by score

`order_reviews` stores 99,224 customer reviews, each with a `review_score`
from 1 (worst) to 5 (best) and an optional free-text `review_comment_message`.
Filtering a numeric column works exactly like filtering a text column — you
just drop the quotes, since `review_score` is stored as an integer, not a
string. Below we check how many 5-star reviews Olist has received; shortly
you'll flip the filter to look at the opposite end of the scale.

In [ ]:
%%sql
-- How many 5-star (best) reviews are there?
SELECT COUNT(*) AS total
FROM order_reviews
WHERE review_score = 5   -- Expected: 57,328

## 5. One more concept + live query — a single aggregate on `order_payments`

You've already used `COUNT(*)` twice today — it's an aggregate function
that collapses every matching row into one number. `AVG()` works the same
way: point it at a numeric column and SQLite collapses the whole column
into a single average, no grouping required (splitting one aggregate into
many per-group values is exactly what `GROUP BY` does — that's next week).
Below we compute the average payment value across all 103,886 rows in
`order_payments`.

In [ ]:
%%sql
-- What is the average payment value across all payments?
SELECT ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM order_payments   -- Expected: 154.10

## Going deeper — dates are stored as TEXT

`order_purchase_timestamp` looks like a date, but SQLite stores it as plain
TEXT in the format `YYYY-MM-DD HH:MM:SS` — there is no real date type
underneath. That means you can't pull out "the year" the way a spreadsheet
date column would let you; instead, SQLite gives you the `strftime()`
function to slice a piece out of a text-formatted date and compare it like
any other string. `strftime('%Y', order_purchase_timestamp)` returns just the
4-digit year as text, which you can then filter on with an ordinary `WHERE`.

In [ ]:
%%sql
-- How many orders were placed in 2017?
SELECT COUNT(*) AS total
FROM orders
WHERE strftime('%Y', order_purchase_timestamp) = '2017'   -- Expected: 45,101

## Common mistake — `LIMIT` without `ORDER BY` is not "the top N"

It's tempting to write `LIMIT 5` and assume you've got "the top 5" rows — but
without an `ORDER BY`, SQLite returns whatever 5 rows it happens to store
first, which is an implementation detail, not a ranking. If you want "the 5
highest" or "the 5 lowest" of anything, `ORDER BY` has to come **before**
`LIMIT` in the query, every single time. This is exactly the trap to watch
for in the group exercise's "top 5" question.

In [ ]:
%%sql
-- ── COMMON MISTAKE ──────────────────────────────────
-- WRONG — LIMIT alone gives 5 arbitrary rows, NOT the 5 highest freight values:
--   SELECT order_id, freight_value FROM order_items LIMIT 5
-- CORRECT — ORDER BY first, then LIMIT:
SELECT order_id, freight_value
FROM order_items
ORDER BY freight_value DESC
LIMIT 5

## Mini-challenge — your turn

⏱ ~5 minutes

Using only `SELECT`, `WHERE`, and `COUNT(*)`, find how many orders have the
status `'unavailable'` — a status distinct from `'canceled'` that means Olist
could not fulfill the order at all.

**Expected:** 609 unavailable orders.

In [ ]:
%%sql
-- ⏱ ~5 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Clause | What it does | Example |
|---|---|---|
| `SELECT` | choose columns | `SELECT customer_id, customer_state FROM customers` |
| `WHERE` | filter rows (text or numeric) | `WHERE payment_type = 'credit_card'` |
| `ORDER BY ... LIMIT` | rank and cap results | `ORDER BY price DESC LIMIT 5` |
| `strftime(...)` | slice a piece out of a TEXT date | `strftime('%Y', order_purchase_timestamp)` |

Today you applied these clauses to four tables you hadn't queried before —
`order_items`, `order_payments`, `customers`, and `order_reviews`. Now it's
time for the **group exercise**: five business questions, one per table,
10 minutes each in your groups — we'll verify every answer together as a
class once everyone's done.

---
**Coming up next week**: GroupBy, Aggregates, and HAVING — instead of
filtering down to individual rows, you'll start summarizing entire groups of
rows (totals per category, averages per state, counts per status) in a
single query.